## Install segmentation-models

In [1]:
# Dependencies are installed in the project's .venv.


# Skip this method
# ! pip install segmentation-models

## Import libraries

The version numbers for the libraries are provided below. It should be noted that other versions may also be compatible, but this list has been tested with the current release. 

* Python : 3.7.9

* segmentation_models : 1.0.1

* Numpy : 1.17.0

* cv2 : 4.5.1

* PIL : 8.0.1

* Keras : 2.3.1

* Tensorflow : 2.0.0

* skimage : 0.17.2



In [2]:
import os
os.environ["SM_FRAMEWORK"] = "tf.keras"
os.environ['KERAS_BACKEND'] = 'tensorflow'

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2
import pandas as pd
import os.path

from scipy import ndimage as ndi
from skimage.segmentation import watershed
from skimage import exposure
from skimage.filters import sobel
from sklearn.metrics import jaccard_score
import random
import time

from keras.models import *
from keras.layers import *
import tensorflow as tf
from keras.optimizers import *
import keras.backend as K
from random import shuffle

from segmentation_models import Unet
from segmentation_models import get_preprocessing

s = 256

Segmentation Models: using `tf.keras` framework.


## User defined functions

In [3]:
def smooth(img):
    return 0.5*img + 0.5*(np.roll(img, +1, axis=0) + np.roll(img, -1, axis=0) +
        np.roll(img, +1, axis=1) + np.roll(img, -1, axis=1) )

def returnIndex(a , value):
    k = np.size(a)
    for i in range(k):
        if(a[i]==value):
            return i
        
def duplex_segmentation(img_path):
    area_frac_duplex=[]
    duplex_image_id=[]
    image = Image.open(img_path).convert('F')
    image = np.copy(np.reshape(np.array(image), image.size[::-1])/255.)   
    image = exposure.equalize_adapthist(image, clip_limit=8.3)
    image = (smooth(smooth(image)))
    image_copy = image
    # markers = np.zeros_like(image)
    markers = np.zeros_like(image, dtype=np.int32)
    markers[image > np.median(image) - 0.10*np.std(image)] = 1     
    markers[image < np.median(image) - 0.10*np.std(image)] = 2
    fig, (ax1) = plt.subplots(1, sharex=True, sharey=True)
    elevation_map = sobel(image)
    # segmentation = morphology.watershed(elevation_map, markers)
    segmentation = watershed(elevation_map, markers)
    segmentation = ndi.binary_fill_holes(segmentation - 1)
    labeled_grains, _ = ndi.label(segmentation)
    plt.close(fig)
    return np.array(segmentation)

def lamellar_segmentation(img_path):
    s = 604
    PIL_image = Image.open(img_path)
    img_in = np.array(PIL_image.convert('L'))
    img_in = cv2.resize(img_in, (s,s), interpolation=cv2.INTER_CUBIC)
    img_in = cv2.blur(img_in, (3,3))
    img_in = cv2.medianBlur(img_in, 3)
    thi, img_in = cv2.threshold(img_in,np.median(img_in),255,cv2.THRESH_BINARY)
    return np.array(img_in)

def read_equi_data(img_mask_path):
    PIL_image = Image.open(img_mask_path[0])
    img_in = np.array(PIL_image.convert('L'))
    img_in = cv2.resize(img_in, (s,s), interpolation=cv2.INTER_CUBIC)
    thi, img_in = cv2.threshold(img_in,np.median(img_in),255,cv2.THRESH_BINARY)
    img_in = 255 - img_in
    img_in = cv2.medianBlur(img_in, 3)
    img_in = np.reshape(img_in, (s,s,1))
    PIL_mask = Image.open(img_mask_path[1])
    mask_in = np.array(PIL_mask.convert('L'))
    # mask_in = cv2.resize(mask_in, (s,s), interpolation=cv2.INTER_CUBIC)
    # thi, mask_in = cv2.threshold(mask_in,np.median(mask_in),255,cv2.THRESH_BINARY)
    # mask_in = mask_in/255
    mask_in = cv2.resize(mask_in, (s, s), interpolation=cv2.INTER_NEAREST)
    mask_in = (mask_in > 127).astype(np.uint8)
    mask_in = mask_in*2
    mask_in = np.reshape(mask_in, (s,s,1))
    # mask_in = tf.keras.utils.to_categorical(mask_in, num_classes = 3, dtype ="uint8")
    mask_in = tf.keras.utils.to_categorical(mask_in, num_classes=3).astype(np.float32)
    return (img_in, mask_in)

def read_lam_data(img_mask_path):
    PIL_image = Image.open(img_mask_path[0])
    img_in = np.array(PIL_image.convert('L'))
    img_in = cv2.resize(img_in, (s,s), interpolation=cv2.INTER_CUBIC)
    img_in = cv2.blur(img_in, (3,3))
    img_in = cv2.medianBlur(img_in, 3)
    img_in = np.reshape(img_in, (s,s,1))
    PIL_mask = Image.open(img_mask_path[1])
    mask_in = np.array(PIL_mask.convert('L'))
    # mask_in = cv2.resize(mask_in, (s,s), interpolation=cv2.INTER_CUBIC)
    # thi, mask_in = cv2.threshold(mask_in,np.median(mask_in),255,cv2.THRESH_BINARY)
    # mask_in = mask_in/255
    mask_in = cv2.resize(mask_in, (s, s), interpolation=cv2.INTER_NEAREST)
    mask_in = (mask_in > 127).astype(np.uint8)
    mask_in = np.reshape(mask_in, (s,s,1))
    # mask_in = tf.keras.utils.to_categorical(mask_in, num_classes = 3, dtype ="uint8")
    mask_in = tf.keras.utils.to_categorical(mask_in, num_classes=3).astype(np.float32)
    return (img_in, mask_in)

## Generating ground truth masks for equiaxed microstructures

In [4]:
num_equi_training_samples = 10
# for i in range(1, num_equi_training_samples + 1):
    # file_id = str(num)
for i in range(1, 21):
    file_id = str(i)
    img_path = 'equi_images/image_' + file_id + '.png'
    res = duplex_segmentation(img_path)
    res_for_cv2 = (res*255.).astype('uint8')
    outfile = 'equi_masks/mask_' + file_id + '.png'
    # cv2.imwrite(outfile, res_for_cv2)
    if not os.path.isfile(outfile):
        if not cv2.imwrite(outfile, res_for_cv2):
            raise OSError(f"Could not save mask: {outfile}")

## Generating ground truth masks for lamellar microstructures

In [5]:
num_lam_training_samples = 10
# for num in range(1, num_lam_training_samples + 1):
for num in range(1, num_lam_training_samples + 10 + 1):
    file_id = str(num)
    img_path = 'lam_images/image_' + file_id + '.png'
    res = lamellar_segmentation(img_path)
    res_for_cv2 = res #(res*255.).astype('uint8')
    outfile = 'lam_masks/mask_' + file_id + '.png'
    # cv2.imwrite(outfile, res_for_cv2)
    if not os.path.isfile(outfile):
        if not cv2.imwrite(outfile, res_for_cv2):
            raise OSError(f"Could not save mask: {outfile}")

## Loading paths for images and masks

The code for the combined loading of images and masks was inspired from https://nanohub.org/tools/imagesegment

In [6]:
train_equi_mask_paths = [("equi_images/image_%d.png"%i, "equi_masks/mask_%d.png"%i) for i in range(1,num_equi_training_samples + 1)] 
            
    
train_lam_mask_paths = [("lam_images/image_%d.png"%i, "lam_masks/mask_%d.png"%i) for i in range(1,num_lam_training_samples + 1)] 


# train_lam_paths = train_lam_mask_paths[int(1):]
# train_equi_paths = train_equi_mask_paths[int(1):]
train_lam_paths = train_lam_mask_paths
train_equi_paths = train_equi_mask_paths
num_train_images = len(train_lam_paths) + len(train_equi_paths)
# num_test_images = len(test_lam_paths) + len(test_equi_paths)

## Define the pre-trained model and specify the weights

In [7]:
BACKBONE = 'resnet34'
preprocess_input = get_preprocessing(BACKBONE)

N = 1

base_model = Unet(BACKBONE, classes = 3, activation='softmax', encoder_weights='imagenet')

inp = Input(shape=(None, None, N))
l1 = Conv2D(3, (1, 1))(inp) 
out = base_model(l1)
#out = BatchNormalization()(out)

model = Model(inp, out, name=base_model.name)

## Compile the model using a preferred loss function and metrics

In [8]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

## Conduct the training process

In [9]:
steps = 0
overall_acc = []
overall_iou = []

out_imgs = np.zeros((num_train_images,)+(s,s)+(1,)) 
out_masks = np.zeros((num_train_images,)+(s,s)+(3,)) 
for i, img_mask_path in enumerate(train_lam_paths):
    img, mask = read_lam_data(img_mask_path) 
    out_imgs[i,...] = img 
    out_masks[i,...] = mask 

flag = len(train_lam_paths)
    
for i, img_mask_path in enumerate(train_equi_paths):
    img, mask = read_equi_data(img_mask_path) 
    out_imgs[i+flag,...] = img 
    out_masks[i+flag,...] = mask 
        

x_train = preprocess_input(out_imgs)

model.fit(x=x_train, y=out_masks, batch_size=3, epochs=5, verbose = 1)

steps += 20

test_equi_mask_paths = [("equi_images/image_%d.png"%i,
                     "equi_masks/mask_%d.png"%i) for i in range(num_equi_training_samples + 1, num_equi_training_samples + 11)] 
                
test_lam_mask_paths = [("lam_images/image_%d.png"%i,
                     "lam_masks/mask_%d.png"%i) for i in range(num_lam_training_samples + 1, num_lam_training_samples + 11)] 
    
num_test_images = len(test_lam_mask_paths) + len(test_equi_mask_paths)
    
out_imgs = np.zeros((num_test_images,)+(s,s)+(1,))
out_masks = np.zeros((num_test_images,)+(s,s)+(3,))
for i, img_mask_path in enumerate(test_lam_mask_paths):
    img, mask = read_lam_data(img_mask_path)
    out_imgs[i,...] = img
    out_masks[i,...] = mask
    
flag = len(test_lam_mask_paths)

for i, img_mask_path in enumerate(test_equi_mask_paths):
    img, mask = read_equi_data(img_mask_path)
    out_imgs[i+flag,...] = img
    out_masks[i+flag,...] = mask

    
prediction = model.predict(preprocess_input(out_imgs.copy()), batch_size=1, verbose=1,)

predicted_classes = prediction.argmax(axis=-1)
true_classes = out_masks.argmax(axis=-1)

curr_round_acc = []
curr_round_iou = []

for i in range(num_test_images):
    foreground_class = 1 if i < len(test_lam_mask_paths) else 2

    truth = true_classes[i] == foreground_class
    predicted = predicted_classes[i] == foreground_class

    accuracy = float(np.mean(true_classes[i] == predicted_classes[i]))
    intersection = np.count_nonzero(truth & predicted)
    union = np.count_nonzero(truth | predicted)

    curr_round_acc.append(accuracy)

    if union > 0:
        curr_round_iou.append(float(intersection / union))

overall_acc.append(curr_round_acc)
overall_iou.append(curr_round_iou)

print("Mean pixel accuracy:", np.mean(curr_round_acc))
print(
    "Mean foreground IoU:",
    np.mean(curr_round_iou) if curr_round_iou else "No foreground to evaluate",
)

Epoch 1/5


1/7 ━━━━━━━━━━━━━━━━━━━━ 1:40 17s/step - accuracy: 0.3076 - loss: 1.5081

2/7 ━━━━━━━━━━━━━━━━━━━━ 2s 497ms/step - accuracy: 0.3754 - loss: 1.2983

3/7 ━━━━━━━━━━━━━━━━━━━━ 1s 484ms/step - accuracy: 0.4025 - loss: 1.2161

4/7 ━━━━━━━━━━━━━━━━━━━━ 1s 481ms/step - accuracy: 0.3737 - loss: 1.2991

...

15/20 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step

17/20 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step

19/20 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step

20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 34ms/step


Mean pixel accuracy: 0.2736793518066406
Mean foreground IoU: 0.2543742031097388
